### **Student Name:** Sadegh Samadi
### **Student ID:** 810102588  
### **Course:** Operating Systems - Fall 1404 - CA3


# Part 1 - Thread Interleaving
---

### Experimental Results

The program was executed twice with $N=5$. The observed outputs are as follows:

**Run #1 Output:**
```text
N = 5
I'm thread 0 of 5
I'm thread 2 of 5
I'm thread 1 of 5
I'm thread 3 of 5
I'm thread 4 of 5
```

**Run #2 Output:**
```text
N = 5
I'm thread 1 of 5
I'm thread 3 of 5
I'm thread 2 of 5
I'm thread 0 of 5
I'm thread 4 of 5
```

### Analysis of Results (Question 5)

**1. Unpredictability of Execution Order**
The output shows that the threads do not finish in the order they were created (0, 1, 2, 3, 4). In Run #2, Thread 1 finished first, while Thread 0 finished fourth.
* **Reason:** The Operating System's Scheduler is responsible for deciding which thread runs at any given moment. The `pthread_create` function merely adds the thread to the scheduler's Ready Queue; it does not guarantee immediate execution.

**2. The Impact of `sleep()`**
The `sleep()` function is the primary factor causing the randomness here.
* **Mechanism:** When a thread calls `sleep()`, it moves from the **Running State** to the **Waiting State**. The CPU performs a context switch to run another thread.
* **Wake Up:** Threads wake up after different durations (0 to 3 seconds). A thread that sleeps for 0 seconds (Thread 1 in Run #2 likely did this) will return to the Ready Queue and execute significantly faster than a thread that sleeps for 3 seconds, regardless of which one was created first.

**3. Preemption and Interleaving**
Even without `sleep()`, the output would likely be non-deterministic due to **Preemption**.
* **Time Slicing:** The OS allocates a specific time slice (quantum) to each thread. If Thread 0's time slice expires before it can print, the OS interrupts it and switches to Thread 1. This results in the **interleaving** of instructions, effectively mixing the execution flow of independent threads.



# Part 1 - Array Summation (Single-thread vs Multi-thread)

### Implementation Logic
* **Single-threaded:** A simple `for` loop calculates the sum on the main thread.
* **Multi-threaded:** The array is partitioned into $T$ blocks. $T$ worker threads calculate partial sums, and the main thread aggregates them.
* **Synchronization:** `pthread_join` is used to wait for all threads to finish.

---

### Experimental Results
The following data was collected from three separate runs of the program with $N=10,000,000$.

| Run | Strategy | Threads ($T$) | Time (Seconds) | Speedup vs Single |
| :--- | :--- | :---: | :---: | :---: |
| **1** | Single-threaded | 1 | 0.019121 | - |
| **1** | Multi-threaded | 2 | 0.025625 | **0.74x (Slower)** |
| | | | | |
| **2** | Single-threaded | 1 | 0.019709 | - |
| **2** | Multi-threaded | 4 | 0.024500 | **0.80x (Slower)** |
| | | | | |
| **3** | Single-threaded | 1 | 0.020038 | - |
| **3** | Multi-threaded | 8 | 0.016239 | **1.23x (Faster)** |

---

### Analysis (Questions 4, 5, 6)
**4. Is Multi-threaded always faster?**
No. As seen in the results, for $T=2$ and $T=4$, the multi-threaded approach was actually **slower** than the single-threaded approach.
* **Reason:** Thread creation (`pthread_create`) and synchronization (`pthread_join`) introduce **System Overhead**. When $T=2$ or $T=4$, the time saved by splitting the calculation was not enough to cover the "cost" of managing those extra threads.

**5. The "Break-even" Point**

| Array Size ($N$) | Threads ($T$) | Single-Threaded (s) | Multi-Threaded (s) | Speedup Factor | Result |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **Small ($10^5$)** | 2 | 0.000383 | 0.001543 | 0.25x | **Single is Faster** |
| **Small ($10^5$)** | 4 | 0.000471 | 0.001148 | 0.41x | **Single is Faster** |
| **Small ($10^5$)** | 8 | 0.000451 | 0.000809 | 0.56x | **Single is Faster** |
| | | | | | |
| **Medium ($10^7$)** | 2 | 0.019908 | 0.031471 | 0.63x | **Single is Faster** |
| **Medium ($10^7$)** | 4 | 0.019092 | 0.027866 | 0.69x | **Single is Faster** |
| **Medium ($10^7$)** | 8 | 0.019784 | 0.018360 | 1.08x | **Multi is Faster** |
| | | | | | |
| **Large ($10^8$)** | 2 | 0.197501 | 0.198651 | 0.99x | Tie |
| **Large ($10^8$)** | 4 | 0.190625 | 0.106578 | **1.79x** | **Multi is Faster** |
| **Large ($10^8$)** | 8 | 0.200278 | 0.136811 | **1.46x** | **Multi is Faster** |

The data clearly shows where multi-threading becomes beneficial:
* **For Small Arrays ($10^5$):** The single-threaded approach is consistently faster (up to 4x faster). The task is so small that the overhead of creating threads dominates the execution time.
* **The Boundary ($10^7$):** This is the transition zone. With fewer threads ($T=2, 4$), the overhead still outweighs the gain. However, at $T=8$, the parallelism finally pays off, slightly beating the single-threaded baseline (0.018s vs 0.019s).
* **For Large Arrays ($10^8$):** Multi-threading provides a significant advantage. At $T=4$, the program runs almost **1.8x faster** than the single-threaded version.
* Interestingly, for the largest array, $T=4$ (0.106s) was actually faster than $T=8$ (0.136s). This suggests that adding more threads isn't always better. It could be due to:
    * **Core Saturation:** If the physical CPU does not have 8 true cores, the OS has to time-slice 8 threads onto fewer cores, increasing context-switching overhead.
    * **Memory Bandwidth:** 8 threads trying to read from RAM simultaneously might bottleneck the memory bus.

**6. Granularity and Overhead (Question 6)**
These results illustrate the concept of **Granularity**—the ratio of computation to communication/overhead.
* **Fine-Grained ($10^5$):** The computation per thread is too small. The system spends more time performing system calls (`pthread_create`, context switching) than actually adding numbers. This explains why the multi-threaded code was slower.
* **Coarse-Grained ($10^8$):** Each thread has a massive chunk of work (25 million additions). The fixed cost of creating the thread becomes negligible compared to the time saved by running on multiple cores simultaneously.

**7. Optional**

**8. OS Differences & Granularity Boundaries**

The "Granularity Boundary" (the array size where multi-threading starts beating single-threading) is not a fixed number; it depends heavily on how the Operating System manages threads.

**1. Thread Implementation Model:**
* **Linux (NPTL - Native POSIX Thread Library):** Linux implements threads as "Lightweight Processes" (LWP). The kernel uses the `clone()` system call, which is highly optimized. Creating a thread in Linux is relatively "cheap" and fast.
* **Windows:** Windows treats threads as distinct kernel objects with more complex data structures (Thread Environment Block - TEB, Kernel Stack, etc.). Creating a thread (`CreateThread`) generally involves more overhead than in Linux.

**2. Scheduler Granularity:**
* **Linux (CFS - Completely Fair Scheduler):** Designed to handle massive numbers of threads efficiently with dynamic time-slicing. It can switch contexts very rapidly.
* **Windows Scheduler:** Historically uses a coarser timer tick (quantum). While efficient, the cost of context switching can be slightly higher due to the heavier thread state.

**Conclusion:**
Because Linux threads are "lighter" to create and manage:
* **On Linux:** The break-even point is **lower**. Multi-threading might become profitable at smaller array sizes (e.g., $N=500,000$).
* **On Windows:** The break-even point is **higher**. You might need a larger array (e.g., $N=2,000,000$) to generate enough "work" to justify the higher "cost" of creating the threads.